# Przywracanie interpunkcji — HerBERT: PolEval i Polish News

**Projekt z Inżynierii Lingwistycznej**

Notatnik dostraja `allegro/herbert-base-cased` do przewidywania znaku po każdym słowie. Łączymy wszystkie dokumenty z `expected.tsv` z pierwszymi 100 000 rekordów `polish-news.csv` (kolumna `content`). Oba źródła przetwarzamy identycznie, a następnie dzielimy wspólny zbiór całymi dokumentami na trening (80%), walidację (10%) i test (10%).

Klasy: `BRAK`, `.`, `,`, `?`, `!`, `:`. Zachowujemy wielkość liter.

## 1. Przygotowanie środowiska

W Colabie lub świeżym środowisku zainstaluj Transformers. Wagi HerBERT zostaną pobrane z Hugging Face przy pierwszym uruchomieniu.


In [1]:
from pathlib import Path
import csv
import copy
import random
import re
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModelForTokenClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urządzenie:", DEVICE)


c:\home\szkoła\drugi stopień\rok 2 semestr 1\INL\projekt\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Urządzenie: cuda


## 2. Etykiety i wczytanie tekstów

Dane wejściowe to słowa bez interpunkcji. Etykieta opisuje znak po słowie. Wspólna funkcja `make_document` przetwarza teksty z `expected.tsv` i kolumny `content` pliku `polish-news.csv`.

Ustaw `DATA_DIR` na katalog z TSV. CSV umieść obok notatnika lub zmień `NEWS_PATH`. `NEWS_DELIMITER` określa separator CSV. `NEWS_LIMIT = 100_000` ogranicza odczyt do pierwszych 100 000 rekordów bez nagłówka; `None` pozwala wczytać cały plik. Puste rekordy wliczają się do limitu, ale dokumenty bez słów są pomijane. Jeden rekord CSV, również wielowierszowy, jest jednym dokumentem.

In [2]:
LABELS = ["BRAK", ".", ",", "?", "!", ":"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

DATA_DIR = Path("data/poleval/train")  # katalog z expected.tsv
if not (DATA_DIR / "expected.tsv").exists():
    for candidate in (Path("2021-punctuation-restoration-main/train"), Path(".")):
        if (candidate / "expected.tsv").exists():
            DATA_DIR = candidate
            break

DELIMITER = ","  # zmień na ";", jeśli plik używa średników

import csv
from itertools import islice

NEWS_PATH = Path("polish-news.csv")
NEWS_LIMIT = 10000  # None = wszystkie rekordy CSV

EXCLAMATION_PATH = Path("zdania_polaczone_4000.csv")
EXCLAMATION_LIMIT = None  # None = wszystkie rekordy CSV

# Długie artykuły mogą przekraczać domyślny limit pola CSV.
csv.field_size_limit(100_000_000)

def read_tsv(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return [row for row in csv.reader(f, delimiter="\t") if row]

def punctuation_in_gap(gap):
    for mark in ["?", "!", ".", ",", ":"]:
        if mark in gap:
            return mark
    return "BRAK"

def make_document(doc_id, expected_text):
    # Te same wyodrębnione słowa tworzą wejście i etykiety.
    matches = list(re.finditer(r"\w+", expected_text, flags=re.UNICODE))
    tokens = [match.group(0) for match in matches]
    labels = []
    for i, match in enumerate(matches):
        gap_end = matches[i + 1].start() if i + 1 < len(matches) else len(expected_text)
        gap = expected_text[match.end():gap_end]
        labels.append(label2id[punctuation_in_gap(gap)])
    return {"id": doc_id, "tokens": tokens, "labels": labels}

expected_path = DATA_DIR / "expected.tsv"
if not expected_path.exists():
    raise FileNotFoundError(f"Nie znaleziono {expected_path}. Ustaw DATA_DIR na katalog zawierający expected.tsv.")

if not NEWS_PATH.exists():
    raise FileNotFoundError(f"Nie znaleziono {NEWS_PATH}. Ustaw NEWS_PATH na plik polish-news.csv.")

poleval_documents = []
for row_number, row in enumerate(read_tsv(expected_path), start=1):
    doc = make_document(f"poleval_{row_number}", " ".join(row).strip())
    if doc["tokens"]:
        doc["source"] = "poleval"
        poleval_documents.append(doc)

news_documents = []
with NEWS_PATH.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter=DELIMITER)
    if not reader.fieldnames or "content" not in reader.fieldnames:
        raise ValueError("CSV musi zawierać kolumnę 'content'. Sprawdź nagłówek i NEWS_DELIMITER.")
    for row_number, row in enumerate(islice(reader, NEWS_LIMIT), start=1):
        content = (row.get("content") or "").strip()
        doc = make_document(f"news_{row_number}", content)
        if doc["tokens"]:
            doc["source"] = "polish-news"
            news_documents.append(doc)


exclamation_documents = []
with EXCLAMATION_PATH.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter=DELIMITER)
    if not reader.fieldnames or "content" not in reader.fieldnames:
        raise ValueError("CSV musi zawierać kolumnę 'content'. Sprawdź nagłówek i EXCLAMATION_DELIMITER.")
    for row_number, row in enumerate(islice(reader, EXCLAMATION_LIMIT), start=1):
        content = (row.get("content") or "").strip()
        doc = make_document(f"exclamation_{row_number}", content)
        if doc["tokens"]:
            doc["source"] = "exclamation"
            exclamation_documents.append(doc)

# Jeden wspólny zbiór, przed podziałem na train/validation/test.
documents = poleval_documents + news_documents + exclamation_documents

print(f"Dokumenty PolEval: {len(poleval_documents)}")
print(f"Wiadomości Polish News: {len(news_documents)}")
print(f"Dokumenty Wykrzyknik: {len(exclamation_documents)}")
print(f"Łącznie dokumentów: {len(documents)}")
print(f"Łącznie słów: {sum(len(doc['tokens']) for doc in documents):,}")
print("Przykładowy tekst wejściowy:", " ".join(documents[0]["tokens"][:12]))
print("Etykiety:", [id2label[i] for i in documents[0]["labels"][:12]])

Dokumenty PolEval: 800
Wiadomości Polish News: 9988
Dokumenty Wykrzyknik: 4000
Łącznie dokumentów: 14788
Łącznie słów: 3,246,633
Przykładowy tekst wejściowy: w wywiadzie dla polski jarosław kaczyński podkreślił że informacje dotyczące radosława sikorskiego
Etykiety: ['BRAK', 'BRAK', 'BRAK', 'BRAK', 'BRAK', 'BRAK', ',', 'BRAK', 'BRAK', 'BRAK', 'BRAK', 'BRAK']


## 3. Podział danych i kontrola etykiet

Najpierw łączymy PolEval i Polish News w `documents`, a następnie dzielimy wspólną listę w proporcji 80/10/10 (z zaokrągleniem liczebności), całymi dokumentami, ze stałym `SEED`. Oba źródła uczestniczą w losowaniu bez wymuszania ich proporcji. Dopiero po podziale tworzymy krótsze fragmenty dla HerBERT, aby fragmenty tego samego dokumentu nie trafiły do różnych zbiorów.

In [3]:
train_docs, temp_docs = train_test_split(documents, test_size=0.2, random_state=SEED)
valid_docs, test_docs = train_test_split(temp_docs, test_size=0.5, random_state=SEED)

for name, docs in (("Trening", train_docs), ("Walidacja", valid_docs), ("Test", test_docs)):
    sources = Counter(doc["source"] for doc in docs)
    print(f"{name}: {len(docs)} dokumentów; źródła: {dict(sources)}")

for name, docs in (("PolEval", poleval_documents), ("Polish News", news_documents), ("Wspólny zbiór", documents), ("Trening", train_docs)):
    counts = Counter(label for doc in docs for label in doc["labels"])
    print(f"\n{name} — etykiety:")
    for i, label in enumerate(LABELS):
        print(f"{label:>5}: {counts.get(i, 0):>10,}")

Trening: 11830 dokumentów; źródła: {'polish-news': 7977, 'exclamation': 3224, 'poleval': 629}
Walidacja: 1479 dokumentów; źródła: {'polish-news': 1001, 'exclamation': 387, 'poleval': 91}
Test: 1479 dokumentów; źródła: {'exclamation': 389, 'polish-news': 1010, 'poleval': 80}

PolEval — etykiety:
 BRAK:    139,173
    .:     10,466
    ,:     10,128
    ?:        797
    !:        118
    ::        912

Polish News — etykiety:
 BRAK:  2,693,241
    .:    184,422
    ,:    176,739
    ?:      1,716
    !:        371
    ::      7,660

Wspólny zbiór — etykiety:
 BRAK:  2,849,034
    .:    194,888
    ,:    187,137
    ?:      4,513
    !:      2,489
    ::      8,572

Trening — etykiety:
 BRAK:  2,258,434
    .:    154,914
    ,:    148,460
    ?:      3,559
    !:      2,017
    ::      6,765


## 4. Tokenizacja słów przez HerBERT

HerBERT może rozbić jedno słowo na kilka tokenów podrzędnych. Etykietę przypisujemy pierwszemu tokenowi danego słowa, a pozostałe tokeny i tokeny specjalne pomijamy w funkcji straty (`-100`). Długie dokumenty dzielimy po 96 słów po wykonaniu podziału train/validation/test, więc fragmenty tego samego dokumentu z żadnego źródła nie trafiają do różnych zbiorów.

In [4]:
MODEL_NAME = "allegro/herbert-base-cased"
MAX_WORDS_PER_CHUNK = 96
MAX_LENGTH = 256
BATCH_SIZE = 4  # przy braku pamięci GPU zmniejsz do 2

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if not tokenizer.is_fast:
    raise RuntimeError("Potrzebny jest szybki tokenizer, który udostępnia word_ids().")

def split_documents_for_model(docs, max_words=MAX_WORDS_PER_CHUNK):
    chunks = []

    def add_chunk(doc_id, tokens, labels):
        encoding = tokenizer(
            tokens,
            is_split_into_words=True,
            add_special_tokens=True,
            truncation=False,
        )

        if len(encoding["input_ids"]) <= MAX_LENGTH:
            chunks.append({
                "id": doc_id,
                "tokens": tokens,
                "labels": labels,
            })
            return

        if len(tokens) == 1:
            raise ValueError(
                f"Pojedyncze słowo w {doc_id} przekracza "
                f"limit {MAX_LENGTH} tokenów: {tokens[0][:100]!r}"
            )

        middle = len(tokens) // 2

        add_chunk(doc_id, tokens[:middle], labels[:middle])
        add_chunk(doc_id, tokens[middle:], labels[middle:])

    for doc in docs:
        for start in range(0, len(doc["tokens"]), max_words):
            tokens = doc["tokens"][start:start + max_words]
            labels = doc["labels"][start:start + max_words]

            if tokens:
                add_chunk(doc["id"], tokens, labels)

    return chunks

train_chunks = split_documents_for_model(train_docs)
valid_chunks = split_documents_for_model(valid_docs)
test_chunks = split_documents_for_model(test_docs)
print(f"Fragmenty modelu — trening: {len(train_chunks)}, walidacja: {len(valid_chunks)}, test: {len(test_chunks)}")

class PunctuationDataset(Dataset):
    def __init__(self, docs):
        self.features = []
        for doc in docs:
            words = doc["tokens"]
            encoding = tokenizer(
                words,
                is_split_into_words=True,
                add_special_tokens=True,
                truncation=True,
                max_length=MAX_LENGTH,
            )
            word_ids = encoding.word_ids()
            aligned_labels = []
            first_subword_word_ids = []
            previous_word_id = None
            for word_id in word_ids:
                if word_id is None:
                    aligned_labels.append(-100)
                    first_subword_word_ids.append(-1)
                elif word_id != previous_word_id:
                    aligned_labels.append(doc["labels"][word_id])
                    first_subword_word_ids.append(word_id)
                else:
                    aligned_labels.append(-100)
                    first_subword_word_ids.append(-1)
                previous_word_id = word_id

            represented_words = {i for i in first_subword_word_ids if i >= 0}
            if len(represented_words) != len(words):
                raise ValueError(
                    f"Tokenizacja ucięła słowa w dokumencie {doc['id']}. "
                    "Zmniejsz MAX_WORDS_PER_CHUNK lub zwiększ MAX_LENGTH."
                )
            self.features.append({
                "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
                "labels": torch.tensor(aligned_labels, dtype=torch.long),
                "word_ids": torch.tensor(first_subword_word_ids, dtype=torch.long),
            })

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

def collate_batch(batch):
    max_len = max(item["input_ids"].numel() for item in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    word_ids = torch.full((len(batch), max_len), -1, dtype=torch.long)
    for i, item in enumerate(batch):
        length = item["input_ids"].numel()
        input_ids[i, :length] = item["input_ids"]
        attention_mask[i, :length] = item["attention_mask"]
        labels[i, :length] = item["labels"]
        word_ids[i, :length] = item["word_ids"]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "word_ids": word_ids}

train_loader = DataLoader(PunctuationDataset(train_chunks), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(PunctuationDataset(valid_chunks), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(PunctuationDataset(test_chunks), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in batch.items()})


Fragmenty modelu — trening: 34122, walidacja: 4438, test: 4392
{'input_ids': (4, 135), 'attention_mask': (4, 135), 'labels': (4, 135), 'word_ids': (4, 135)}


## 5. Model HerBERT

Warstwa klasyfikacyjna przewiduje jedną z sześciu etykiet dla pierwszego tokenu każdego słowa. Inicjalizujemy ją dla tego zadania, a wagi enkodera pobieramy z `allegro/herbert-base-cased`.


In [5]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)
print("Model:", MODEL_NAME)
print("Liczba parametrów:", f"{sum(p.numel() for p in model.parameters()):,}")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at allegro/herbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: allegro/herbert-base-cased
Liczba parametrów: 123,856,902


## 6. Trening i early stopping

Używamy ważonej straty `CrossEntropyLoss`, pomijając tokeny z etykietą `-100`. Wagi klas obliczamy tylko z danych treningowych. Early stopping obserwuje `val_loss`, zatrzymuje trening po trzech epokach bez poprawy i przywraca wagi z najlepszej epoki.

In [6]:

import numpy as np
import torch.nn as nn

num_labels = model.config.num_labels

# Zliczamy etykiety w zbiorze treningowym; -100 oznacza tokeny pomijane.
counts = np.zeros(num_labels, dtype=np.int64)


for example in train_loader.dataset:
    labels = np.asarray(example["labels"], dtype=np.int64)
    labels = labels[labels != -100]  # pomijamy etykiety paddingu
    counts += np.bincount(labels, minlength=num_labels)

weights = 1.0 / np.sqrt(np.maximum(counts, 1))
weights = weights / weights.mean()

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=DEVICE,
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100,
)

for label_id, count, weight in zip( range(num_labels), counts, weights):
    print(f"etykieta {label_id}: liczba={count}, waga={weight:.3f}")

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.set_grad_enabled(training):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            loss = criterion(
                outputs.logits.reshape(-1, num_labels),
                labels.reshape(-1),
            )

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        batches += 1

    return total_loss / max(batches, 1)

EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 1e-4
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_model_state = copy.deepcopy(model.state_dict())

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    valid_loss = run_epoch(valid_loader, training=False)
    print(f"Epoka {epoch:02d} | train loss: {train_loss:.4f} | val loss: {valid_loss:.4f}")
    if valid_loss < best_val_loss - MIN_DELTA:
        best_val_loss = valid_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_without_improvement += 1
        print(f"Brak poprawy walidacyjnej: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping — zatrzymuję trening.")
            break

model.load_state_dict(best_model_state)
print(f"Przywrócono wagi z epoki {best_epoch} (najlepszy val loss: {best_val_loss:.4f}).")


etykieta 0: liczba=2258434, waga=0.070
etykieta 1: liczba=154914, waga=0.267
etykieta 2: liczba=148460, waga=0.273
etykieta 3: liczba=3559, waga=1.765
etykieta 4: liczba=2017, waga=2.344
etykieta 5: liczba=6765, waga=1.280
Epoka 01 | train loss: 0.1800 | val loss: 0.1293
Epoka 02 | train loss: 0.1185 | val loss: 0.1374
Brak poprawy walidacyjnej: 1/3
Epoka 03 | train loss: 0.0947 | val loss: 0.1620
Brak poprawy walidacyjnej: 2/3
Epoka 04 | train loss: 0.0784 | val loss: 0.1881
Brak poprawy walidacyjnej: 3/3
Early stopping — zatrzymuję trening.
Przywrócono wagi z epoki 1 (najlepszy val loss: 0.1293).


## 7. Ewaluacja na wspólnym zbiorze testowym

Test to odłożona część połączonych danych PolEval i Polish News, a nie oficjalny test PolEval. Raport pokazuje precision, recall i F1 dla etykiet, Macro F1 i Weighted F1. Predykcje i prawdziwe etykiety porównujemy na poziomie słów, nie tokenów podrzędnych. Do early stopping służy wyłącznie walidacja.

In [7]:
def predict_loader(loader):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            pred = outputs.logits.argmax(dim=-1).cpu()
            word_ids = batch["word_ids"]
            labels = batch["labels"]
            for i in range(pred.size(0)):
                positions = word_ids[i] >= 0
                all_true.extend(labels[i, positions].tolist())
                all_pred.extend(pred[i, positions].tolist())
    return all_true, all_pred

y_true, y_pred = predict_loader(test_loader)
all_labels = list(range(len(LABELS)))
print(classification_report(
    y_true, y_pred, labels=all_labels, target_names=LABELS, zero_division=0, digits=3
))
print("Macro F1:", round(f1_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0), 4))
print("Weighted F1:", round(f1_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0), 4))


              precision    recall  f1-score   support

        BRAK      0.995     0.977     0.986    293544
           .      0.941     0.974     0.957     19901
           ,      0.773     0.936     0.847     18849
           ?      0.559     0.862     0.679       458
           !      0.891     0.850     0.870       240
           :      0.538     0.760     0.630       899

    accuracy                          0.974    333891
   macro avg      0.783     0.893     0.828    333891
weighted avg      0.978     0.974     0.975    333891

Macro F1: 0.8282
Weighted F1: 0.9751


## 8. Predykcja dla własnego zdania

Funkcja wyodrębnia słowa tak samo jak przygotowanie danych, zachowuje ich oryginalną wielkość liter i przewiduje znak po każdym słowie.


In [8]:
def predict_words(tokens):
    if not tokens:
        return []
    predictions = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(tokens), MAX_WORDS_PER_CHUNK):
            words = tokens[start:start + MAX_WORDS_PER_CHUNK]
            encoded = tokenizer(
                words, is_split_into_words=True, add_special_tokens=True,
                truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
            )
            word_ids = encoded.word_ids(batch_index=0)
            inputs = {key: value.to(DEVICE) for key, value in encoded.items()}
            logits = model(**inputs).logits[0].argmax(dim=-1).cpu().tolist()
            chunk_predictions = [None] * len(words)
            for token_position, word_id in enumerate(word_ids):
                if word_id is not None and chunk_predictions[word_id] is None:
                    chunk_predictions[word_id] = logits[token_position]
            if any(value is None for value in chunk_predictions):
                raise ValueError("Nie udało się uzyskać predykcji dla wszystkich słów; zmniejsz długość fragmentu.")
            predictions.extend(chunk_predictions)
    return list(zip(tokens, [id2label[i] for i in predictions]))

def predict_sentence(sentence):
    tokens = [m.group(0) for m in re.finditer(r"\w+", sentence, flags=re.UNICODE)]
    return predict_words(tokens)

def format_prediction(predictions):
    out = ""
    for token, label in predictions:
        mark = "" if label == "BRAK" else label
        out += token + mark + " "
    return out.strip()

example = "Kiedy wrócisz daj mi znać"
predictions = predict_sentence(example)
print(predictions)
print(format_prediction(predictions))


[('Kiedy', 'BRAK'), ('wrócisz', ','), ('daj', 'BRAK'), ('mi', 'BRAK'), ('znać', '!')]
Kiedy wrócisz, daj mi znać!


## 9. Zapis modelu

In [9]:
model.save_pretrained("models/herbert")
tokenizer.save_pretrained("models/herbert")

('models/herbert\\tokenizer_config.json',
 'models/herbert\\special_tokens_map.json',
 'models/herbert\\vocab.json',
 'models/herbert\\merges.txt',
 'models/herbert\\added_tokens.json',
 'models/herbert\\tokenizer.json')